In [27]:
import requests
import pandas as pd
import json
from datetime import datetime

In [28]:
date = '2026-06-29'         # action(1/4): 输入查询的日期

In [29]:
spot_code = '588000.SH'     # action(2/4): 输入期权链标的代码
option_EX = '2607'          # action(3/4)：输入到期月份：'YYMM'

# action(4/4): 更新ifind token，刷新后access token有效期为7天
accessToken = '286c4713719c00645acce6dbe7f0b03bc10f4d8a.signs_ODgzMDE1NzQ5'

---

In [30]:
opt_key = str(spot_code) + '_' + str(option_EX) # 588000.SH_2606
start_date = date # 'YYYY-MM-DD'
end_date = start_date

opt_code = ''
opt_var = [
    'ths_open_interest_option',
    'ths_position_chg_option',
    'ths_settle_option',
    'ths_implied_volatility_option',

    'ths_delta_option',
    'ths_gamma_option',
    'ths_theta_option',
    'ths_vega_option',
    'ths_rho_option',

    'ths_delta_exchg_option',
    'ths_gamma_exchg_option',
    'ths_theta_exchg_option',
    'ths_vega_exchg_option',
    'ths_rho_exchg_option',

    ]

underlying_var = [
    'ths_underlying_volatility_30d_option',
    'ths_underlying_volatility_60d_option',
    'ths_underlying_volatility_90d_option',
    'ths_contract_multiplier',
    'ths_surplus_duration_calendar_option',
    'ths_remain_trade_days',
    'ths_last_strike_date_option',
    ]


spot_var = ['ths_close_price_fund']

In [31]:
multiplier = 10_000
Spot = 1.0

In [32]:
def load_option_config(config_path="option_code_map.json"):
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)

config = load_option_config()
code_map = config[opt_key]["code_to_name"]

opt_code_str = ",".join(code_map.keys())

if not opt_code_str:
    raise ValueError("期权代码为空，请检查spot_code 输入或配置文件opt_list.json")


In [33]:

opt_list_temp = opt_code_str.split(",")
opt_list_temp[1]

'10011736.SH'

In [34]:
RT_Url = 'https://quantapi.51ifind.com/api/v1/date_sequence'
thsHeaders = {"Content-Type":"application/json","access_token":accessToken}

thsPara_spot = {"codes":spot_code,
                'startdate':start_date,
                'enddate': end_date,
                'indipara': [{"indicator": x, "indiparams": []} for x in spot_var]
                }

thsResponse_spot = requests.post(url=RT_Url, json=thsPara_spot, headers=thsHeaders)
data_spot = thsResponse_spot.json()

thsPara_opt = {"codes":opt_code_str,
               'startdate':start_date,
               'enddate': end_date,
               'indipara': [{"indicator": x, "indiparams": []} for x in opt_var]
               }

thsResponse = requests.post(url=RT_Url, json=thsPara_opt, headers=thsHeaders)
data_opt = thsResponse.json()

thsPara_underying = {'codes':opt_list_temp[1],
               'startdate':start_date,
               'enddate': end_date,
               'indipara': [{"indicator": x, "indiparams": []} for x in underlying_var]
               }

thsResponse = requests.post(url=RT_Url, json=thsPara_underying, headers=thsHeaders)
data_underying = thsResponse.json()

In [35]:
Spot = float(data_spot["tables"][0]["table"]["ths_close_price_fund"][0])
# multiplier = int(data_opt["tables"][0]["table"]["ths_contract_multiplier"][0])

rows_opt = []
for item in data_opt['tables']:
    code = item['thscode']
    oi = item['table']['ths_open_interest_option'][0]
    d_oi = item['table']['ths_position_chg_option'][0]
    settle = item['table']['ths_settle_option'][0]
    iv = item['table']['ths_implied_volatility_option'][0]

    delta = item['table']['ths_delta_option'][0]
    gamma = item['table']['ths_gamma_option'][0]
    theta = item['table']['ths_theta_option'][0]
    vega = item['table']['ths_vega_option'][0]
    rho = item['table']['ths_rho_option'][0]

    delta_exchg = item['table']['ths_delta_exchg_option'][0]
    gamma_exchg = item['table']['ths_gamma_exchg_option'][0]
    theta_exchg = item['table']['ths_theta_exchg_option'][0]
    vega_exchg = item['table']['ths_vega_exchg_option'][0]
    rho_exchg = item['table']['ths_rho_exchg_option'][0]

    rows_opt.append({
        'thscode': code,
        'oi': oi,
        'd_oi': d_oi,
        'settle': settle,
        'iv': iv,

        'delta': delta,
        'gamma': gamma,
        'theta': theta,
        'vega': vega,
        'rho': rho,

        'delta_exchg': delta_exchg,
        'gamma_exchg': gamma_exchg,
        'theta_exchg': theta_exchg,
        'vega_exchg': vega_exchg,
        'rho_exchg': rho_exchg,
        })

df_opt = pd.DataFrame(rows_opt)

rows_underlying = []
for item in data_underying['tables']:
    code = item['thscode']
    hv30 = item['table']['ths_underlying_volatility_30d_option'][0]
    hv60 = item['table']['ths_underlying_volatility_60d_option'][0]
    hv90 = item['table']['ths_underlying_volatility_90d_option'][0]
    mul =item['table']['ths_contract_multiplier'][0]
    DTE_n = item['table']['ths_surplus_duration_calendar_option'][0]
    DTE_t = item['table']['ths_remain_trade_days'][0]
    last_date = item['table']['ths_last_strike_date_option'][0]

    rows_underlying.append({
        'thscode': code,
        'hv30': hv30,
        'hv60': hv60,
        'hv90': hv90,
        'mul': mul,
        'DTE_n': DTE_n,
        'DTE_t': DTE_t,
        'last_date': last_date
    })

df_underlying = pd.DataFrame(rows_underlying)


In [36]:
if df_underlying['mul'][0] == None:
    df_underlying['mul'] = 10_000

In [37]:
df_opt['name'] = df_opt["thscode"].map(code_map)
df_opt.insert(loc=0, column='ETF_close', value = Spot)
df_opt.insert(loc=0, column='date', value = date)

df_opt.insert(loc=len(df_opt.columns), column='hv30', value = df_underlying['hv30'][0])
df_opt.insert(loc=len(df_opt.columns), column='hv60', value = df_underlying['hv60'][0])
df_opt.insert(loc=len(df_opt.columns), column='hv90', value = df_underlying['hv90'][0])
df_opt.insert(loc=len(df_opt.columns), column='mul', value = df_underlying['mul'][0])
df_opt.insert(loc=len(df_opt.columns), column='DTE_n', value = df_underlying['DTE_n'][0])
df_opt.insert(loc=len(df_opt.columns), column='DTE_t', value = df_underlying['DTE_t'][0])
df_opt.insert(loc=len(df_opt.columns), column='last_date', value = df_underlying['last_date'][0])

In [38]:
dt = datetime.strptime(date, "%Y-%m-%d")
MMDD = dt.strftime("%m%d")
save_to_path = f'ifind_raw_data/{MMDD}_588000SH2607.csv'
print(save_to_path)

df_opt.to_csv(save_to_path, index=False)

ifind_raw_data/0629_588000SH2607.csv


In [39]:
df_opt.info()
# action(6+1/6): 通过这里的显示结果预览数据
# 偶数行； 成对出现的数据缺口可以接受（当前日期，某K价合约未上市）

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 25 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         38 non-null     object 
 1   ETF_close    38 non-null     float64
 2   thscode      38 non-null     object 
 3   oi           38 non-null     int64  
 4   d_oi         38 non-null     int64  
 5   settle       38 non-null     float64
 6   iv           38 non-null     float64
 7   delta        38 non-null     float64
 8   gamma        38 non-null     float64
 9   theta        38 non-null     float64
 10  vega         38 non-null     float64
 11  rho          38 non-null     float64
 12  delta_exchg  38 non-null     float64
 13  gamma_exchg  38 non-null     float64
 14  theta_exchg  38 non-null     float64
 15  vega_exchg   38 non-null     float64
 16  rho_exchg    38 non-null     float64
 17  name         38 non-null     object 
 18  hv30         38 non-null     float64
 19  hv60      